# Phase 2 — Xây dựng và kiểm chứng QUBO

**Nhánh A — NT409 Task Placement.** Dùng lại dữ liệu và bộ đánh giá Phase 1.

Mục tiêu: chuyển `min_active_hosts` cùng ràng buộc one-hot/CPU/RAM thành QUBO,
kiểm tra công thức năng lượng và đối chiếu nghiệm tối ưu với Phase 1.

Chọn kernel **nt409-khiem (Python 3.12)** trong VS Code. Chạy từng cell từ trên xuống.
Notebook chỉ đọc bốn JSON hiện có; cuối notebook tạo `data/reference_results/phase2_qubo.json`.
Các hàm xây dựng mô hình nằm trong `src/qubo.py`; diễn giải đầy đủ ở `docs/qubo_formulation.md`.

## Bước 1 — Kiểm tra môi trường và nhập module

In [1]:
import json
import sys
from pathlib import Path
from itertools import product, combinations

import numpy as np
import pandas as pd
import qiskit
import qiskit_optimization
from IPython.display import display

cwd = Path.cwd().resolve()
ROOT = next((p for p in [cwd, *cwd.parents]
             if (p / "requirements.txt").is_file()
             and (p / "notebooks").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open this notebook inside nt409-task-placement.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.placement import evaluate_model, evaluate_placement
from src.qubo import (
    build_qubo, slack_weights, complete_with_best_slack,
    decode_qubo, energy_parts, exact_qubo_reference,
)

print("Project root:", ROOT)
print("Python:", sys.executable)
print("Qiskit:", qiskit.__version__)
print("Qiskit Optimization:", qiskit_optimization.__version__)

Project root: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement
Python: c:\Users\Khiem\anaconda3\envs\nt409-khiem\python.exe
Qiskit: 2.5.2
Qiskit Optimization: 0.7.0


**Giải thích:** notebook tự tìm thư mục gốc để import `src` và đọc dữ liệu bằng đường dẫn
ổn định. Trên máy bạn, đường dẫn Python cần trỏ vào môi trường `nt409-khiem`.
Module mới đã được kiểm tra với Qiskit 2.5.2 và Qiskit Optimization 0.7.0.
Nếu vừa sửa module sau khi import, hãy Restart Kernel rồi chạy lại từ đầu.

## Bước 2 — Đọc lại dữ liệu Phase 1

In [2]:
instance_ids = ["p0_reference", "p1_choice", "p1_tight", "p1_infeasible"]
instances = {
    name: json.loads((ROOT / "data" / "instances" / f"{name}.json").read_text(encoding="utf-8"))
    for name in instance_ids
}
for name, instance in instances.items():
    assert instance["instance_id"] == name
    assert instance["objective"] == "min_active_hosts"

expected = {
    "p0_reference": {"feasible": 1, "objective": 2, "optimal": 1, "bits": 23, "energy": 2},
    "p1_choice": {"feasible": 11, "objective": 1, "optimal": 1, "bits": 20, "energy": 1},
    "p1_tight": {"feasible": 4, "objective": 2, "optimal": 4, "bits": 18, "energy": 2},
    "p1_infeasible": {"feasible": 0, "objective": None, "optimal": 0, "bits": 20, "energy": 5},
}
display(pd.DataFrame([
    {"instance_id": name, "tasks": len(inst["tasks"]), "hosts": len(inst["hosts"])}
    for name, inst in instances.items()
]))

,instance_id,tasks,hosts
0,p0_reference,4,2
1,p1_choice,4,2
2,p1_tight,4,2
3,p1_infeasible,4,2


**Giải thích:** giữ nguyên đầu vào để so sánh với Phase 1. `expected` là đáp án kiểm tra,
không được dùng để dựng hàm năng lượng hay chọn nghiệm. Nếu thuật toán chuyển đổi sai,
các `assert` ở bước sau phải báo lỗi thay vì sửa đáp án mong đợi cho khớp.

## Bước 3 — Hiểu biến slack và dựng mô hình

Với một tài nguyên, đặt `usage = tổng nhu cầu các task trên host`:

$$usage\le C y \quad\Longleftrightarrow\quad usage+s-Cy=0,\quad 0\le s\le C.$$

Ví dụ host bật, C=5, usage=4 thì s=1. Khi usage=6, không có s không âm nào bù được.
Slack biểu diễn phần tài nguyên chưa dùng, không biểu diễn một phần task.

Ta dùng bit để biểu diễn slack. Ví dụ C=5 dùng trọng số `[1,2,2]`:
`s=b0+2*b1+2*b2`, nhận đủ giá trị 0..5. Có thể có nhiều cách mã hóa cùng một giá trị.

In [3]:
display(pd.DataFrame([
    {"capacity": c, "weights": slack_weights(c), "slack_bits": len(slack_weights(c))}
    for c in [3, 5, 6, 8]
]))

bundles = {name: build_qubo(inst) for name, inst in instances.items()}
reference_bundle = bundles["p0_reference"]
print(reference_bundle["model"].prettyprint())

,capacity,weights,slack_bits
0,3,"[1, 2]",2
1,5,"[1, 2, 2]",3
2,6,"[1, 2, 3]",3
3,8,"[1, 2, 4, 1]",4


Problem name: p0_reference

Minimize
  y_0 + y_1

Subject to
  Linear constraints (8)
    x_0_0 + x_0_1 == 1  'assign_0'
    x_1_0 + x_1_1 == 1  'assign_1'
    x_2_0 + x_2_1 == 1  'assign_2'
    x_3_0 + x_3_1 == 1  'assign_3'
    s_0_cpu_0 + 2*s_0_cpu_1 + 2*s_0_cpu_2 + 2*x_0_0 + 3*x_1_0 + x_2_0 + 4*x_3_0
    - 5*y_0 == 0  'capacity_0_cpu'
    s_0_ram_0 + 2*s_0_ram_1 + 3*s_0_ram_2 + 4*x_0_0 + 2*x_1_0 + 3*x_2_0
    + 5*x_3_0 - 6*y_0 == 0  'capacity_0_ram'
    s_1_cpu_0 + 2*s_1_cpu_1 + 3*s_1_cpu_2 + 2*x_0_1 + 3*x_1_1 + x_2_1 + 4*x_3_1
    - 6*y_1 == 0  'capacity_1_cpu'
    s_1_ram_0 + 2*s_1_ram_1 + 4*s_1_ram_2 + s_1_ram_3 + 4*x_0_1 + 2*x_1_1
    + 3*x_2_1 + 5*x_3_1 - 8*y_1 == 0  'capacity_1_ram'

  Binary variables (23)
    x_0_0 x_0_1 x_1_0 x_1_1 x_2_0 x_2_1 x_3_0 x_3_1 y_0 y_1 s_0_cpu_0 s_0_cpu_1
    s_0_cpu_2 s_0_ram_0 s_0_ram_1 s_0_ram_2 s_1_cpu_0 s_1_cpu_1 s_1_cpu_2
    s_1_ram_0 s_1_ram_1 s_1_ram_2 s_1_ram_3



**Giải thích:** `model` chứa mục tiêu `sum(y)` và các phương trình ràng buộc.
Với mỗi task, có một phương trình gán đúng một host. Với mỗi host, có hai phương trình
CPU và RAM kèm slack. Bốn task/hai host tạo **8 phương trình**.
`build_qubo()` dùng chính `_arrays()` của Phase 1 để kiểm tra dữ liệu CPU/RAM đầu vào.

## Bước 4 — Chuyển phương trình thành penalty

Hàm năng lượng là:

$$E=\sum_h y_h
+P\sum_i(\sum_h x_{i,h}-1)^2
+P\sum_{h,r}(\sum_i d_{i,r}x_{i,h}+s_{h,r}-C_{h,r}y_h)^2.$$

`LinearEqualityToPenalty(penalty=P)` trong `src/qubo.py` khai triển các bình phương.
Vì mọi biến đều là bit và các biểu thức trong ngoặc là tuyến tính, E là đa thức bậc tối đa 2.

Chọn **P=m+1=3**. Mục tiêu gốc không quá m=2, còn vi phạm phương trình với dữ liệu nguyên
bị phạt ít nhất P=3. Do đó, nếu có nghiệm hợp lệ, mọi cực tiểu toàn cục phải hợp lệ.
Đây là bảo đảm toán học cho mô hình này, chưa phải bảo đảm QAOA tìm được cực tiểu.

In [4]:
size_rows = []
for name, bundle in bundles.items():
    model, qubo = bundle["model"], bundle["qubo"]
    assert qubo.get_num_linear_constraints() == 0
    assert qubo.get_num_quadratic_constraints() == 0
    assert qubo.get_num_binary_vars() == qubo.get_num_vars()
    assert qubo.get_num_vars() == expected[name]["bits"]
    size_rows.append({
        "instance_id": name,
        "base_bits": bundle["base_count"],
        "slack_bits": qubo.get_num_vars() - bundle["base_count"],
        "qubo_bits": qubo.get_num_vars(),
        "constraints_before": model.get_num_linear_constraints(),
        "constraints_after": qubo.get_num_linear_constraints(),
        "penalty": bundle["penalty"],
    })
display(pd.DataFrame(size_rows).set_index("instance_id"))

,base_bits,slack_bits,qubo_bits,constraints_before,constraints_after,penalty
instance_id,,,,,,
p0_reference,10,13,23,8,0,3.0
p1_choice,10,10,20,8,0,3.0
p1_tight,10,8,18,8,0,3.0
p1_infeasible,10,10,20,8,0,3.0


**Kết quả cần thấy:** số bit QUBO lần lượt là **23, 20, 18, 20**; số ràng buộc tường minh
sau chuyển đổi bằng 0. Ràng buộc vẫn được thể hiện bằng penalty trong năng lượng.

10 bit của Phase 1 chỉ là x,y. Các bit bổ sung là slack. Với ánh xạ trực tiếp một bit
thành một qubit ở Phase 3, số qubit sẽ bằng tổng bit QUBO ở đây.

## Bước 5 — So sánh năng lượng của các phương án quen thuộc

In [5]:
def base_from_result(result):
    return np.asarray(result["x"], dtype=int).ravel().tolist() + result["y"]

ref = instances["p0_reference"]
choice = instances["p1_choice"]
valid = evaluate_placement(ref, [0, 0, 1, 1])
x_twice = np.asarray(valid["x"]).copy()
x_twice[0] = [1, 1]
cases = [
    ("p0_reference", "valid", valid),
    ("p0_reference", "overloaded", evaluate_placement(ref, [0, 0, 0, 1])),
    ("p0_reference", "T1_on_two_hosts", evaluate_model(ref, x_twice, valid["y"])),
    ("p0_reference", "H0_disabled", evaluate_model(ref, valid["x"], [0, 1])),
    ("p1_choice", "packed", evaluate_placement(choice, [0, 0, 0, 0])),
    ("p1_choice", "spread", evaluate_placement(choice, [0, 0, 1, 1])),
]
case_rows = []
for name, case, original in cases:
    bundle = bundles[name]
    bits = complete_with_best_slack(bundle, base_from_result(original))
    parts = energy_parts(bundle, bits)
    energy = float(bundle["qubo"].objective.evaluate(bits))
    assert np.isclose(energy, parts["energy"])
    assert decode_qubo(bundle, bits)["feasible"] == original["feasible"]
    case_rows.append({
        "case": case, "feasible": original["feasible"],
        "objective": original["objective"], **parts,
    })
case_table = pd.DataFrame(case_rows).set_index("case")
case_table["objective"] = case_table["objective"].astype("Int64")
display(case_table)

,feasible,objective,active_hosts,assignment_error,capacity_error,energy
case,,,,,,
valid,True,2,2,0,0,2.0
overloaded,False,<NA>,2,0,10,32.0
T1_on_two_hosts,False,<NA>,2,1,17,56.0
H0_disabled,False,<NA>,1,0,61,184.0
packed,True,1,1,0,0,1.0
spread,True,2,2,0,0,2.0


**Giải thích:** `assignment_error` và `capacity_error` là tổng bình phương sai số,
không phải số lượng ràng buộc vi phạm. Năng lượng bằng
`active_hosts + P*(assignment_error + capacity_error)`.

Với phương án hợp lệ và slack đúng, hai phần sai số bằng 0: năng lượng bằng số host bật.
Do đó `packed` có năng lượng 1, `spread` có năng lượng 2; cả hai vẫn gán mỗi task cho đúng một host.
Ba phương án sai của p0 lần lượt có năng lượng 32, 56, 184. `objective=<NA>` nghĩa là
phương án không hợp lệ; vẫn có thể tính năng lượng QUBO của nó.

## Bước 6 — Vì sao penalty không được quá nhỏ?

In [6]:
strong = bundles["p1_choice"]
weak = build_qubo(choice, penalty=0.1)
packed_base = base_from_result(evaluate_placement(choice, [0, 0, 0, 0]))
empty_base = [0] * strong["base_count"]
penalty_rows = []
for label, bundle in [("P=3", strong), ("P=0.1", weak)]:
    for case, base in [("packed", packed_base), ("no_tasks_assigned", empty_base)]:
        bits = complete_with_best_slack(bundle, base)
        penalty_rows.append({
            "penalty": label, "case": case,
            "feasible": decode_qubo(bundle, bits)["feasible"],
            "energy": float(bundle["qubo"].objective.evaluate(bits)),
        })
display(pd.DataFrame(penalty_rows))
assert penalty_rows[3]["energy"] < penalty_rows[2]["energy"]

,penalty,case,feasible,energy
0,P=3,packed,True,1.0
1,P=3,no_tasks_assigned,False,12.0
2,P=0.1,packed,True,1.0
3,P=0.1,no_tasks_assigned,False,0.4


**Giải thích:** không gán task nào và tắt cả hai host vi phạm bốn ràng buộc one-hot.
Với P=0.1, năng lượng chỉ là `4*0.1=0.4`, thấp hơn phương án hợp lệ dùng một host (E=1).
Với P=3, phương án sai đó có E=12. Đây là lý do phải kiểm chứng hệ số phạt trước khi chạy QAOA.
Ví dụ này không sửa mô hình chính: các `bundles` chính vẫn dùng P=3.

## Bước 7 — Kiểm tra khai triển QUBO

Hai biểu thức đang so sánh đều là đa thức bậc tối đa 2 theo các bit.
Giá trị tại vector 0 xác định hằng số; từng vector có một bit 1 xác định các hệ số bậc một;
từng vector có hai bit 1 xác định các hệ số tương tác. Vì bit bình phương bằng chính nó,
kiểm tra các điểm này đủ để xác nhận hai đa thức giống nhau trên toàn miền nhị phân.

In [7]:
coefficient_checks = {}
for name, bundle in bundles.items():
    qubo = bundle["qubo"]
    count = qubo.get_num_vars()
    supports = [()] + [(i,) for i in range(count)] + list(combinations(range(count), 2))
    for support in supports:
        bits = np.zeros(count, dtype=int)
        bits[list(support)] = 1
        actual = float(qubo.objective.evaluate(bits))
        formula = energy_parts(bundle, bits)["energy"]
        assert np.isclose(actual, formula, rtol=0, atol=1e-8), (name, support)
    coefficient_checks[name] = len(supports)
print("PASS — equality of QUBO and squared-penalty formula:", coefficient_checks)

PASS — equality of QUBO and squared-penalty formula: {'p0_reference': 277, 'p1_choice': 211, 'p1_tight': 172, 'p1_infeasible': 211}


**Giải thích:** bước này kiểm tra phép chuyển đổi và thứ tự bit kể cả trên các vector không hợp lệ.
Việc chỉ thử một phương án hợp lệ sẽ không phát hiện được nhiều lỗi ở các số hạng penalty.

## Bước 8 — Tìm cực tiểu QUBO và đối chiếu Phase 1

Không chỉ duyệt 16 placement: chúng đã thỏa one-hot, nên không thể phát hiện trường hợp
QUBO ưu tiên bỏ sót hoặc gán trùng task. Ta duyệt **toàn bộ 1024 tổ hợp x,y**.

Với mỗi x,y cố định, chọn từng slack độc lập:

$$s^*_{h,r}=\max(0,C_{h,r}y_h-usage_{h,r}).$$

Giá trị này tối thiểu hóa bình phương sai số, luôn thuộc 0..C và mã hóa được bằng bit.
Vì thế cách kiểm chứng tìm được cực tiểu toàn cục của QUBO mà không cần duyệt mọi bit slack.
Đây là bộ kiểm chứng cổ điển cho fixture nhỏ; không phải thuật toán QAOA.

In [8]:
references = {}
validation_rows = []
for name, bundle in bundles.items():
    inst, exp = instances[name], expected[name]
    # Independent Phase 1 reference: enumerate canonical placements.
    feasible = []
    for placement in product(range(bundle["m"]), repeat=bundle["n"]):
        result = evaluate_placement(inst, placement)
        if result["feasible"]:
            feasible.append(result)
    best_original = min((r["objective"] for r in feasible), default=None)
    optimal_original = sum(r["objective"] == best_original for r in feasible)
    assert (len(feasible), best_original, optimal_original) == (
        exp["feasible"], exp["objective"], exp["optimal"]
    ), name

    exact = exact_qubo_reference(bundle)
    references[name] = exact
    assert np.isclose(exact["minimum_energy"], exp["energy"]), name
    if best_original is None:
        assert all(not item["decoded"]["feasible"] for item in exact["minimizers"])
        assert exact["minimum_energy"] > bundle["m"]
    else:
        assert all(item["decoded"]["feasible"] for item in exact["minimizers"])
        assert all(item["decoded"]["objective"] == best_original
                   for item in exact["minimizers"])
        assert len(exact["minimizers"]) == optimal_original
        assert np.isclose(exact["minimum_energy"], best_original)

    validation_rows.append({
        "instance_id": name,
        "base_bits": bundle["base_count"],
        "slack_bits": bundle["qubo"].get_num_vars() - bundle["base_count"],
        "qubo_bits": bundle["qubo"].get_num_vars(),
        "penalty": bundle["penalty"],
        "checked_base_vectors": exact["checked_base_vectors"],
        "min_energy": exact["minimum_energy"],
        "original_best_objective": best_original,
        "minimizer_xy_count": len(exact["minimizers"]),
        "all_minimizers_feasible": all(v["decoded"]["feasible"] for v in exact["minimizers"]),
        "passed": True,
    })
validation = pd.DataFrame(validation_rows).set_index("instance_id")
validation["original_best_objective"] = validation["original_best_objective"].astype("Int64")
display(validation)
print("PASS — all four QUBO models agree with Phase 1.")

,base_bits,slack_bits,qubo_bits,penalty,checked_base_vectors,min_energy,original_best_objective,minimizer_xy_count,all_minimizers_feasible,passed
instance_id,,,,,,,,,,
p0_reference,10,13,23,3.0,1024,2.0,2,1,True,True
p1_choice,10,10,20,3.0,1024,1.0,1,1,True,True
p1_tight,10,8,18,3.0,1024,2.0,2,4,True,True
p1_infeasible,10,10,20,3.0,1024,5.0,<NA>,12,False,True


PASS — all four QUBO models agree with Phase 1.


**Kết quả cần đạt:** min_energy lần lượt là **2, 1, 2, 5**.
Ba bộ có nghiệm giữ đúng số host tối ưu là **2, 1, 2**.
`p1_infeasible` có `all_minimizers_feasible=False` nhưng vẫn `passed=True` vì đó là kết quả đúng.

QUBO luôn có cực tiểu trên tập bit hữu hạn. Cực tiểu E=5 của bộ vô nghiệm không có nghĩa dùng 5 host:
ví dụ bật 2 host và bỏ sót 1 task thì năng lượng là `2 + 3 = 5`.
`minimizer_xy_count` đếm theo x,y; không đếm các chuỗi slack khác nhau biểu diễn cùng một giá trị.

## Bước 9 — Giải mã một nghiệm tối ưu

In [9]:
name = "p1_choice"
bundle = bundles[name]
best = references[name]["minimizers"][0]
decoded = decode_qubo(bundle, best["bits"])
assert decoded["feasible"]

display(pd.DataFrame(
    decoded["x"],
    index=[t["id"] for t in instances[name]["tasks"]],
    columns=[h["id"] for h in instances[name]["hosts"]],
))
print("y:", decoded["y"])
print("Original objective:", decoded["objective"])
print("QUBO energy:", best["energy"])
print("Energy parts:", energy_parts(bundle, best["bits"]))

,H0,H1
T1,1,0
T2,1,0
T3,1,0
T4,1,0


y: [1, 0]
Original objective: 1
QUBO energy: 1.0
Energy parts: {'active_hosts': 1, 'assignment_error': 0, 'capacity_error': 0, 'energy': 1.0}


**Giải thích:** cả bốn task nằm trên H0, `y=[1,0]`, objective=1 và E=1.
Hàm giải mã vẫn gọi `evaluate_model()` của Phase 1, giữ nguyên y từ nghiệm QUBO.
Sau này kết quả QAOA cũng phải đi qua bước kiểm tra này trước khi công nhận là lời giải hợp lệ.

## Bước 10 — Lưu kết quả kiểm chứng để bàn giao

In [10]:
report = {
    "schema_version": 1,
    "objective": "min_active_hosts",
    "method": "enumerate_all_xy_and_minimize_slack_exactly",
    "qiskit_version": qiskit.__version__,
    "qiskit_optimization_version": qiskit_optimization.__version__,
    "results": [],
}
for row in validation_rows:
    name = row["instance_id"]
    bundle = bundles[name]
    best = references[name]["minimizers"][0]
    report["results"].append({
        **row,
        "coefficient_checks": coefficient_checks[name],
        "variable_names": [v.name for v in bundle["qubo"].variables],
        "example_minimizer": best,
    })
report_path = ROOT / "data" / "reference_results" / "phase2_qubo.json"
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False), encoding="utf-8")
print("Saved:", report_path)

Saved: C:\Users\Khiem\quang-tum-com-pu-ting\nt409-task-placement\data\reference_results\phase2_qubo.json


**Giải thích:** đây là báo cáo đáp án chuẩn của Phase 2, không phải benchmark QAOA.
C có thể nối với các kết quả sau bằng `instance_id`. Luôn phân biệt `min_energy`
của QUBO và `original_best_objective` của bài toán gốc.

## Trước khi commit

1. Restart Kernel rồi Run All; không có lỗi và các assert đều đạt.
2. Lưu notebook bằng Ctrl+S để giữ output trên máy bạn.
3. Commit `src/qubo.py`, `notebooks/03_qubo_model.ipynb`, `docs/qubo_formulation.md`
   và `data/reference_results/phase2_qubo.json`.
4. Push lên nhánh A và mở PR vào main để B/C có thể cập nhật.

**Phân công:** A tiếp tục Ising/QAOA sau khi Phase 2 này đạt; B xây dựng heuristic/MILP;
C chuẩn hóa bảng kết quả và thiết kế benchmark. B không cần dùng biến slack của QUBO.

Nguồn API:
- [Qiskit Optimization 0.7.0 — Converters](https://qiskit-community.github.io/qiskit-optimization/tutorials/02_converters_for_quadratic_programs.html)
- [LinearEqualityToPenalty](https://qiskit-community.github.io/qiskit-optimization/stubs/qiskit_optimization.converters.LinearEqualityToPenalty.html)